# 第 7 章 · Trajectory、序列化与可观测性

**这一章你会得到什么**：知道一次 Agent 运行留下的“轨迹”里到底存了什么、存到哪，以及 harness 为什么要在**每一步之后**都保存。这是评测、复现和 debug 的基础。

## 📖 对照源码（在 IDE 里打开这些文件，边看边跑）

- `src/minisweagent/agents/default.py` **L157–178** — `serialize()`（打包整段轨迹）
- `src/minisweagent/agents/default.py` **L180–188** — `save()`（写 JSON 文件）
- `src/minisweagent/agents/default.py` **L118–119** — `run()` 里 `finally: save(...)`（崩了也留现场）
- `src/minisweagent/utils/serialize.py` **L6–29** — `recursive_merge`（多方信息合并）

> 快捷：代码格里 `函数名??` 直接打印源码；或用 `show_source("相对路径", 起始行, 结束行)`。

In [1]:
import os, sys
from pathlib import Path
os.environ["MSWEA_SILENT_STARTUP"] = "1"
REPO = Path(r"/Users/xinranzhao/Documents/llm-study/books/mini-swe-agent-source-guide/mini-swe-agent")
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.chdir(REPO)
import minisweagent
print("mini-SWE-agent:", minisweagent.__version__)

mini-SWE-agent: 2.4.5


In [2]:
def show_source(rel_path: str, start: int, end: int) -> None:
    lines = (REPO / rel_path).read_text().splitlines()
    end = min(end, len(lines))
    w = len(str(end))
    for n in range(start, end + 1):
        print(f"{n:>{w}}  {lines[n - 1]}")

## 实验 1：跑一个两步 Agent，序列化它的状态

`serialize()` 把 messages、成本、调用次数、配置、退出状态打包成一个可存 JSON 的 dict。

In [5]:
from minisweagent.agents.default import DefaultAgent
from minisweagent.environments.local import LocalEnvironment
from minisweagent.models.test_models import DeterministicToolcallModel, make_toolcall_output

first = {"command": "echo hi", "tool_call_id": "c1"}
submit = {"command": "echo COMPLETE_TASK_AND_SUBMIT_FINAL_OUTPUT\necho final", "tool_call_id": "c2"}
agent = DefaultAgent(
    DeterministicToolcallModel(outputs=[
        make_toolcall_output("看一下", [], [first]),
        make_toolcall_output("提交", [], [submit]),
    ]),
    LocalEnvironment(cwd=str(REPO)),
    system_template="system", instance_template="{{task}}", cost_limit=5,
)
agent.run("演示轨迹")
data = agent.serialize()
for key in data:
    print(data)
print("顶层 keys:", list(data.keys()))
print("info keys:", list(data["info"].keys()))
print("exit_status:", data["info"]["exit_status"])
print("submission:", repr(data["info"]["submission"]))
print("api_calls:", data["info"]["model_stats"]["api_calls"])
print("消息条数:", len(data["messages"]))

{'info': {'model_stats': {'instance_cost': 2.0, 'api_calls': 2}, 'config': {'agent': {'system_template': 'system', 'instance_template': '{{task}}', 'step_limit': 0, 'cost_limit': 5.0, 'wall_time_limit_seconds': 0, 'max_consecutive_format_errors': 3, 'output_path': None}, 'agent_type': 'minisweagent.agents.default.DefaultAgent', 'model': {'outputs': [{'role': 'assistant', 'content': '看一下', 'tool_calls': [], 'extra': {'actions': [{'command': 'echo hi', 'tool_call_id': 'c1'}], 'cost': 1.0, 'timestamp': 1784997138.2528121}}, {'role': 'assistant', 'content': '提交', 'tool_calls': [], 'extra': {'actions': [{'command': 'echo COMPLETE_TASK_AND_SUBMIT_FINAL_OUTPUT\necho final', 'tool_call_id': 'c2'}], 'cost': 1.0, 'timestamp': 1784997138.252813}}], 'model_name': 'deterministic_toolcall', 'cost_per_call': 1.0, 'observation_template': '{% if output.exception_info %}<exception>{{output.exception_info}}</exception>\n{% endif %}<returncode>{{output.returncode}}</returncode>\n<output>\n{{output.output}

## 观察点
- 轨迹自带 `trajectory_format` 版本号——评测和复现依赖它稳定。
- `exit_status` / `submission` 取自最后一条 `exit` 消息的 `extra`（回忆第 4 章）。
- `serialize()` 用 `recursive_merge` 把 agent / model / environment 三方的信息合成一棵树。

## 实验 2：保存到文件再读回来

`save(path)` 会把序列化结果写成 JSON。我们存到一个临时文件，读回确认。

In [6]:
import json, tempfile
from pathlib import Path
tmp = Path(tempfile.mkdtemp()) / "traj.json"
agent.save(tmp)
loaded = json.loads(tmp.read_text())
print("文件存在:", tmp.exists())
print("读回的 exit_status:", loaded["info"]["exit_status"])
print("读回的消息角色:", [m["role"] for m in loaded["messages"]])
tmp.unlink(); tmp.parent.rmdir()  # 清理临时文件

文件存在: True
读回的 exit_status: Submitted
读回的消息角色: ['system', 'user', 'assistant', 'tool', 'assistant', 'exit']


## 实验 3：为什么保存在 `finally` 里？

回忆 `run()`：`self.save(...)` 在 `try/except/finally` 的 **finally** 分支。
意味着即使某一步抛出未预期异常，**崩溃前的轨迹也已落盘**。跑下面这格感受一下。

In [7]:
show_source("src/minisweagent/agents/default.py", 96, 122)

 96          while True:
 97              try:
 98                  self.step()
 99                  self.n_consecutive_format_errors = 0  # reset on any clean step
100              except FormatError as e:
101                  # The call was billed before parsing failed, so query() never got to charge it.
102                  self.cost += e.messages[0].get("extra", {}).get("cost", 0.0)
103                  self.n_consecutive_format_errors += 1
104                  if 0 < self.config.max_consecutive_format_errors <= self.n_consecutive_format_errors:
105                      self.add_messages(
106                          *e.messages,
107                          {
108                              "role": "exit",
109                              "content": "RepeatedFormatError",
110                              "extra": {"exit_status": "RepeatedFormatError", "submission": ""},
111                          },
112                      )
113                  else:
114                      

## 实验 4：`recursive_merge` 的合并语义

轨迹树是多方信息递归合并出来的。亲手验证“后者覆盖前者、嵌套 dict 递归合并”。

In [15]:
from minisweagent.utils.serialize import recursive_merge
a = {"info": {"config": {"x": 1}, "cost": 10.0}}
b = {"info": {"config": {"y": 2}, "cost": 3.0}}
print(recursive_merge(a, b))

{'info': {'config': {'x': 1, 'y': 2}, 'cost': 3.0}}


## 动手：从轨迹里抽取“这次跑成功了吗”

补全一个小函数：输入 `serialize()` 的结果，返回 `(是否成功, 提交内容)`。
提示：成功的判据是 `info.exit_status == "Submitted"`。

In [24]:
def summarize(traj: dict):
    # TODO: 从 traj["info"] 里取 exit_status 和 submission
    return (traj["info"].get("exit_status", ""), traj["info"]["submission"])
    # TODO: return (exit_status == "Submitted", submission)
    ...

# 参考实现（先自己写上面，再对照）：
def summarize_ref(traj: dict):
    info = traj["info"]
    return info["exit_status"] == "Submitted", info["submission"]
summarize(data)
print(summarize(data))

('Submitted', 'final\n')


## 闭卷检查
1. 轨迹里至少存了哪几类信息？
2. `exit_status` 从哪来？
3. 为什么 `save()` 要放在 `finally`？这对评测意味着什么？

**完成标准**：你能说清“一次运行 = 一条可存可读的 trajectory”，并知道它是评测与复现的接口。